In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create project folders
base_path = '/content/drive/MyDrive/llm_calibration'
os.makedirs(f'{base_path}/results', exist_ok=True)

print("✅ Google Drive mounted successfully!")
print("📁 Folder structure created at:", base_path)
print("\nFolders created:")
for folder in os.listdir(base_path):
    print(f"  └── {folder}")

Mounted at /content/drive
✅ Google Drive mounted successfully!
📁 Folder structure created at: /content/drive/MyDrive/llm_calibration

Folders created:
  └── results
  └── figures


In [2]:
!pip install transformers datasets accelerate -q

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import requests
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Check GPU
print("🖥️ Device Info:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\n🐍 Torch version: {torch.__version__}")

# Fixed seed - same every time
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n🌱 Random seed fixed at: {SEED}")

# Base path
base_path = '/content/drive/MyDrive/llm_calibration'
print(f"📁 Base path: {base_path}")

🖥️ Device Info:
  CUDA available: True
  GPU: Tesla T4
  Memory: 15.6 GB

🐍 Torch version: 2.10.0+cu128

🌱 Random seed fixed at: 42
📁 Base path: /content/drive/MyDrive/llm_calibration


In [3]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
base_path = '/content/drive/MyDrive/llm_calibration'
pile_save_path = f'{base_path}/pile_50k.csv'

# Check if already downloaded
if os.path.exists(pile_save_path):
    print("✅ PILE dataset already exists on Drive, skipping download.")
    pile_df = pd.read_csv(pile_save_path)
    print(f"   Loaded {len(pile_df)} samples")
    print(f"   Columns: {pile_df.columns.tolist()}")
    print(f"\nSample text preview:")
    print(pile_df['text'].iloc[0][:200])
else:
    print("📥 Downloading PILE dataset (streaming)...")
    dataset = load_dataset(
        "monology/pile-uncopyrighted",
        split="train",
        streaming=True,
        trust_remote_code=True
    )

    samples = []
    target = 50000

    for i, example in enumerate(dataset):
        if len(samples) >= target:
            break
        text = example.get('text', '').strip()
        if len(text) > 100:  # filter very short texts
            samples.append({'text': text})

        if (i+1) % 5000 == 0:
            print(f"  Processed {i+1} raw samples, collected {len(samples)} valid...")

    pile_df = pd.DataFrame(samples[:target])
    pile_df.to_csv(pile_save_path, index=False)

    print(f"\n✅ PILE dataset saved!")
    print(f"   Total samples: {len(pile_df)}")
    print(f"   Saved to: {pile_save_path}")
    print(f"\nSample text preview:")
    print(pile_df['text'].iloc[0][:200])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'monology/pile-uncopyrighted' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'monology/pile-uncopyrighted' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Downloading PILE dataset (streaming)...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

  Processed 5000 raw samples, collected 4965 valid...
  Processed 10000 raw samples, collected 9930 valid...
  Processed 15000 raw samples, collected 14903 valid...
  Processed 20000 raw samples, collected 19871 valid...
  Processed 25000 raw samples, collected 24831 valid...
  Processed 30000 raw samples, collected 29791 valid...
  Processed 35000 raw samples, collected 34758 valid...
  Processed 40000 raw samples, collected 39722 valid...
  Processed 45000 raw samples, collected 44686 valid...
  Processed 50000 raw samples, collected 49661 valid...

✅ PILE dataset saved!
   Total samples: 50000
   Saved to: /content/drive/MyDrive/llm_calibration/pile_50k.csv

Sample text preview:
It is done, and submitted. You can play “Survival of the Tastiest” on Android, and on the web. Playing on the web works, but you have to simulate multi-touch for table moving and that can be a bit con


In [9]:
import pandas as pd

base_path = '/content/drive/MyDrive/llm_calibration'
trex_df = pd.read_csv(f'{base_path}/trex_50k.csv')

print(f"Total samples: {len(trex_df)}")
print(f"\nTop 10 most common relations:")
print(trex_df['relation'].value_counts().head(10))
print(f"\nTotal unique relations: {trex_df['relation'].nunique()}")
print(f"Total unique subjects: {trex_df['subject'].nunique()}")
print(f"\nRandom 5 samples:")
print(trex_df.sample(5, random_state=42)[['text','entity']])

Total samples: 50000

Top 10 most common relations:
relation
country                                             7574
instance of                                         4915
located in the administrative territorial entity    4608
country of citizenship                              4066
occupation                                          3383
taxon rank                                          3365
sport                                               2674
place of birth                                      2147
parent taxon                                        1602
performer                                           1389
Name: count, dtype: int64

Total unique relations: 106
Total unique subjects: 49136

Random 5 samples:
                                                    text             entity
33553                  Venezuela's shares border with is             France
9427   Carrollton's located in the administrative ter...        New Orleans
199                 Ruth Sawtell Walli

In [12]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
base_path = '/content/drive/MyDrive/llm_calibration'
pile_save_path = f'{base_path}/pile_50k.csv'

# Delete old file
if os.path.exists(pile_save_path):
    os.remove(pile_save_path)
    print("🗑️ Deleted old PILE file")

# We will collect from 3 different parts:
# Part 1: first 17k (skip 0)
# Part 2: middle 17k (skip 200k)
# Part 3: later 16k (skip 500k)
parts = [
    {'skip': 0,      'target': 17000, 'label': 'Start'},
    {'skip': 200000, 'target': 17000, 'label': 'Middle'},
    {'skip': 500000, 'target': 16000, 'label': 'End'},
]

all_samples = []

for part in parts:
    print(f"\n📥 Collecting from {part['label']} (skip={part['skip']})...")

    dataset = load_dataset(
        "monology/pile-uncopyrighted",
        split="train",
        streaming=True
    )

    samples = []
    skipped = 0
    processed = 0

    for example in dataset:
        # Skip to desired position
        if skipped < part['skip']:
            skipped += 1
            continue

        text = example.get('text', '').strip()
        if len(text) > 100:
            samples.append({
                'text': text,
                'source': example.get('meta', {}).get('pile_set_name', 'unknown')
            })
            processed += 1

        if processed >= part['target']:
            break

        if processed % 5000 == 0 and processed > 0:
            print(f"   Collected {processed}/{part['target']}...")

    print(f"   ✅ Collected {len(samples)} samples from {part['label']}")
    all_samples.extend(samples)

# Shuffle
print(f"\n🔀 Shuffling {len(all_samples)} total samples...")
import random
random.seed(SEED)
random.shuffle(all_samples)

pile_df = pd.DataFrame(all_samples[:50000])
pile_df.to_csv(pile_save_path, index=False)

print(f"\n✅ PILE dataset rebuilt!")
print(f"   Total samples: {len(pile_df)}")
print(f"   Saved to: {pile_save_path}")
print(f"\n📊 Source distribution:")
print(pile_df['source'].value_counts().head(10))
print(f"\nSample previews:")
for i in range(3):
    print(f"\n  Sample {i+1} [{pile_df['source'].iloc[i]}]:")
    print(f"    {pile_df['text'].iloc[i][:150]}")

🗑️ Deleted old PILE file

📥 Collecting from Start (skip=0)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/17000...
   Collected 10000/17000...
   Collected 15000/17000...
   ✅ Collected 17000 samples from Start

📥 Collecting from Middle (skip=200000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/17000...
   Collected 10000/17000...
   Collected 15000/17000...
   ✅ Collected 17000 samples from Middle

📥 Collecting from End (skip=500000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/16000...
   Collected 10000/16000...
   Collected 15000/16000...
   ✅ Collected 16000 samples from End

🔀 Shuffling 50000 total samples...

✅ PILE dataset rebuilt!
   Total samples: 50000
   Saved to: /content/drive/MyDrive/llm_calibration/pile_50k.csv

📊 Source distribution:
source
Pile-CC              14827
StackExchange         8417
PubMed Abstracts      8263
Github                4925
Wikipedia (en)        4866
USPTO Backgrounds     3198
PubMed Central        1550
FreeLaw               1390
ArXiv                  679
DM Mathematics         580
Name: count, dtype: int64

Sample previews:

  Sample 1 [Pile-CC]:
    Like most websites we uses cookies. In order to deliver a personalised, responsive service and to improve the site, we remember and store information 

  Sample 2 [Wikipedia (en)]:
    Hector Beers

Hector George Beers was an English cricketer active from 1914 to 1921 who played for Northamptonshire (Northants). He was born in Potter

  Sample 3 [PubMed A

In [13]:
from datasets import load_dataset
import pandas as pd

print("📥 Loading MMLU dataset...")

# MMLU has 57 subjects, we load all of them
mmlu = load_dataset("cais/mmlu", "all", split="test")

print(f"\n✅ MMLU loaded!")
print(f"   Total samples: {len(mmlu)}")
print(f"   Features: {mmlu.features}")

# Convert to dataframe for easy handling
mmlu_df = pd.DataFrame(mmlu)
print(f"\n📊 Subject distribution (top 10):")
print(mmlu_df['subject'].value_counts().head(10))
print(f"\n   Total unique subjects: {mmlu_df['subject'].nunique()}")

# Preview
print(f"\n📝 Sample preview:")
sample = mmlu_df.iloc[0]
print(f"   Question: {sample['question']}")
print(f"   Choices: {sample['choices']}")
print(f"   Answer:  {sample['answer']} ({sample['choices'][sample['answer']]})")
print(f"   Subject: {sample['subject']}")

# Also load validation set for 5-shot examples
mmlu_val = load_dataset("cais/mmlu", "all", split="validation")
mmlu_val_df = pd.DataFrame(mmlu_val)
print(f"\n✅ MMLU validation set (for 5-shot): {len(mmlu_val_df)} samples")

📥 Loading MMLU dataset...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]


✅ MMLU loaded!
   Total samples: 14042
   Features: {'question': Value('string'), 'subject': Value('string'), 'choices': List(Value('string')), 'answer': ClassLabel(names=['A', 'B', 'C', 'D'])}

📊 Subject distribution (top 10):
subject
professional_law              1534
moral_scenarios                895
miscellaneous                  783
professional_psychology        612
high_school_psychology         545
high_school_macroeconomics     390
elementary_mathematics         378
moral_disputes                 346
prehistory                     324
philosophy                     311
Name: count, dtype: int64

   Total unique subjects: 57

📝 Sample preview:
   Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
   Choices: ['0', '4', '2', '6']
   Answer:  1 (4)
   Subject: abstract_algebra

✅ MMLU validation set (for 5-shot): 1531 samples


In [14]:
import pandas as pd

base_path = '/content/drive/MyDrive/llm_calibration'

mmlu_save_path = f'{base_path}/mmlu_test.csv'
mmlu_val_save_path = f'{base_path}/mmlu_val.csv'

# Save test set
mmlu_df.to_csv(mmlu_save_path, index=False)
print(f"✅ MMLU test set saved: {len(mmlu_df)} samples")

# Save validation set (for 5-shot)
mmlu_val_df.to_csv(mmlu_val_save_path, index=False)
print(f"✅ MMLU validation set saved: {len(mmlu_val_df)} samples")

# Verify
print(f"\n📁 Files on Drive:")
import os
for f in os.listdir(base_path):
    size = os.path.getsize(f'{base_path}/{f}') / 1e6
    print(f"   {f} — {size:.1f} MB")

✅ MMLU test set saved: 14042 samples
✅ MMLU validation set saved: 1531 samples

📁 Files on Drive:
   results — 0.0 MB
   figures — 0.0 MB
   trex_50k.csv — 4.3 MB
   pile_50k.csv — 277.7 MB
   mmlu_test.csv — 6.8 MB
   mmlu_val.csv — 0.7 MB


In [15]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

# ============================================================
# ECE CALCULATION
# ============================================================
def compute_ece(confidences, accuracies, n_bins=10):
    """
    Compute Expected Calibration Error (ECE)
    Lower ECE = better calibration
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_samples = len(confidences)

    for i in range(n_bins):
        low = bin_boundaries[i]
        high = bin_boundaries[i + 1]

        # Find samples in this bin
        in_bin = np.where(
            (confidences > low) & (confidences <= high)
        )[0]

        if len(in_bin) == 0:
            continue

        bin_acc = np.mean(accuracies[in_bin])
        bin_conf = np.mean(confidences[in_bin])
        bin_weight = len(in_bin) / total_samples

        ece += bin_weight * abs(bin_acc - bin_conf)

    return ece

# ============================================================
# TASK 1: CAUSAL LANGUAGE MODELING (PILE)
# ============================================================
def evaluate_clm(model, tokenizer, pile_df, n_samples=1000, max_length=128):
    """
    Task: predict next token at a random position in text
    Returns: ECE, Accuracy
    """
    model.eval()
    confidences = []
    accuracies = []

    # Sample n_samples from pile
    samples = pile_df.sample(n=n_samples, random_state=SEED)

    with torch.no_grad():
        for idx, row in samples.iterrows():
            text = row['text']
            tokens = tokenizer.encode(
                text,
                return_tensors='pt',
                max_length=max_length,
                truncation=True
            )

            if tokens.shape[1] < 10:
                continue

            # Random position (not first or last token)
            pos = np.random.randint(5, tokens.shape[1] - 1)

            # Input = tokens up to pos
            input_ids = tokens[:, :pos].to(device)
            true_token = tokens[0, pos].item()

            # Get model prediction
            outputs = model(input_ids)
            logits = outputs.logits[0, -1, :]  # last position logits

            # Softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)
            pred_token = torch.argmax(probs).item()
            confidence = probs[pred_token].item()

            is_correct = int(pred_token == true_token)

            confidences.append(confidence)
            accuracies.append(is_correct)

    confidences = np.array(confidences)
    accuracies = np.array(accuracies)

    ece = compute_ece(confidences, accuracies)
    acc = np.mean(accuracies)

    return ece, acc, len(confidences)

# ============================================================
# TASK 2: FACTS GENERATION (T-REx)
# ============================================================
def evaluate_facts(model, tokenizer, trex_df, n_samples=1000):
    """
    Task: predict first token of entity given subject + relation
    Returns: ECE, Accuracy
    """
    model.eval()
    confidences = []
    accuracies = []

    samples = trex_df.sample(n=n_samples, random_state=SEED)

    with torch.no_grad():
        for idx, row in samples.iterrows():
            text = row['text']       # "Serge Blisko's occupation is"
            entity = row['entity']   # "politician"

            # Get first token of entity
            entity_tokens = tokenizer.encode(
                ' ' + entity,  # space before entity for proper tokenization
                add_special_tokens=False
            )
            if not entity_tokens:
                continue
            true_first_token = entity_tokens[0]

            # Tokenize input text
            input_ids = tokenizer.encode(
                text,
                return_tensors='pt',
                max_length=64,
                truncation=True
            ).to(device)

            # Get model prediction
            outputs = model(input_ids)
            logits = outputs.logits[0, -1, :]
            probs = torch.softmax(logits, dim=-1)

            pred_token = torch.argmax(probs).item()
            confidence = probs[pred_token].item()
            is_correct = int(pred_token == true_first_token)

            confidences.append(confidence)
            accuracies.append(is_correct)

    confidences = np.array(confidences)
    accuracies = np.array(accuracies)

    ece = compute_ece(confidences, accuracies)
    acc = np.mean(accuracies)

    return ece, acc, len(confidences)

# ============================================================
# TASK 3: MULTI-TASK LANGUAGE UNDERSTANDING (MMLU)
# ============================================================
def format_mmlu_prompt(question, choices, subject, few_shot_examples):
    """Format MMLU question with 5-shot examples"""
    choice_labels = ['A', 'B', 'C', 'D']
    subject_clean = subject.replace('_', ' ')

    prompt = f"The following are multiple choice questions (with answers) about {subject_clean}.\n\n"

    # Add 5-shot examples
    for ex in few_shot_examples:
        prompt += f"Question: {ex['question']}\n"
        for i, choice in enumerate(ex['choices']):
            prompt += f"{choice_labels[i]}. {choice}\n"
        prompt += f"Answer: {choice_labels[ex['answer']]}\n\n"

    # Add actual question
    prompt += f"Question: {question}\n"
    for i, choice in enumerate(choices):
        prompt += f"{choice_labels[i]}. {choice}\n"
    prompt += "Answer:"

    return prompt

def evaluate_mmlu(model, tokenizer, mmlu_df, mmlu_val_df, n_samples=500):
    """
    Task: multiple choice QA with 5-shot prompting
    Returns: ECE, Accuracy
    """
    model.eval()
    confidences = []
    accuracies = []

    choice_labels = ['A', 'B', 'C', 'D']

    # Get token ids for A, B, C, D
    choice_token_ids = []
    for label in choice_labels:
        tokens = tokenizer.encode(f' {label}', add_special_tokens=False)
        choice_token_ids.append(tokens[0])

    # Sample questions
    samples = mmlu_df.sample(n=min(n_samples, len(mmlu_df)), random_state=SEED)

    with torch.no_grad():
        for idx, row in samples.iterrows():
            subject = row['subject']

            # Get 5-shot examples from validation set for same subject
            subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
            if len(subject_val) >= 5:
                few_shots = subject_val.sample(5, random_state=SEED).to_dict('records')
            elif len(subject_val) > 0:
                few_shots = subject_val.to_dict('records')
            else:
                # Use any 5 from validation
                few_shots = mmlu_val_df.sample(
                    min(5, len(mmlu_val_df)),
                    random_state=SEED
                ).to_dict('records')

            prompt = format_mmlu_prompt(
                row['question'],
                row['choices'],
                subject,
                few_shots
            )

            input_ids = tokenizer.encode(
                prompt,
                return_tensors='pt',
                max_length=1024,
                truncation=True
            ).to(device)

            outputs = model(input_ids)
            logits = outputs.logits[0, -1, :]

            # Get probabilities only for A, B, C, D tokens
            choice_logits = torch.tensor(
                [logits[tid].item() for tid in choice_token_ids]
            )
            probs = torch.softmax(choice_logits, dim=-1)

            pred_idx = torch.argmax(probs).item()
            confidence = probs[pred_idx].item()
            is_correct = int(pred_idx == row['answer'])

            confidences.append(confidence)
            accuracies.append(is_correct)

    confidences = np.array(confidences)
    accuracies = np.array(accuracies)

    ece = compute_ece(confidences, accuracies)
    acc = np.mean(accuracies)

    return ece, acc, len(confidences)

print("✅ All evaluation functions defined!")
print("\nFunctions ready:")
print("  → compute_ece(confidences, accuracies)")
print("  → evaluate_clm(model, tokenizer, pile_df)")
print("  → evaluate_facts(model, tokenizer, trex_df)")
print("  → evaluate_mmlu(model, tokenizer, mmlu_df, mmlu_val_df)")

🖥️ Using device: cuda
✅ All evaluation functions defined!

Functions ready:
  → compute_ece(confidences, accuracies)
  → evaluate_clm(model, tokenizer, pile_df)
  → evaluate_facts(model, tokenizer, trex_df)
  → evaluate_mmlu(model, tokenizer, mmlu_df, mmlu_val_df)


In [16]:
import pandas as pd
import os

base_path = '/content/drive/MyDrive/llm_calibration'

# Load all datasets from Drive
print("📂 Loading datasets from Drive...")
pile_df = pd.read_csv(f'{base_path}/pile_50k.csv')
trex_df = pd.read_csv(f'{base_path}/trex_50k.csv')
mmlu_df = pd.read_csv(f'{base_path}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{base_path}/mmlu_val.csv')

# Fix MMLU choices column (saved as string, need to convert back to list)
import ast
mmlu_df['choices'] = mmlu_df['choices'].apply(ast.literal_eval)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(ast.literal_eval)

print(f"✅ PILE loaded:      {len(pile_df):,} samples")
print(f"✅ T-REx loaded:     {len(trex_df):,} samples")
print(f"✅ MMLU test:        {len(mmlu_df):,} samples")
print(f"✅ MMLU validation:  {len(mmlu_val_df):,} samples")

# ============================================================
# PHASE 1 MODEL LIST
# ============================================================
PHASE1_MODELS = [
    "EleutherAI/pythia-70m",
    "EleutherAI/pythia-160m",
    "EleutherAI/pythia-410m",
    "EleutherAI/pythia-1b",
    "EleutherAI/pythia-1.4b",
]

# ============================================================
# MAIN EVALUATION FUNCTION
# ============================================================
def run_evaluation(model_name, revision=None, n_clm=1000,
                   n_facts=1000, n_mmlu=500):
    """
    Load model and run all 3 evaluations
    Returns dict of results
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import gc

    print(f"\n{'='*60}")
    print(f"🤖 Model: {model_name}")
    if revision:
        print(f"📌 Revision: {revision}")
    print(f"{'='*60}")

    # Load tokenizer and model
    print("📥 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        revision=revision if revision else "main"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("📥 Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=revision if revision else "main",
        torch_dtype=torch.float32,
        device_map='auto'
    )
    model.eval()

    # Count parameters
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Model loaded! Parameters: {n_params:.0f}M")
    print(f"   GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    results = {
        'model': model_name,
        'revision': revision if revision else 'final',
        'n_params_M': round(n_params, 1)
    }

    # Task 1: CLM
    print(f"\n📊 Task 1: Causal Language Modeling (n={n_clm})...")
    ece_clm, acc_clm, n_clm_actual = evaluate_clm(
        model, tokenizer, pile_df, n_samples=n_clm
    )
    results['ece_clm'] = round(ece_clm, 4)
    results['acc_clm'] = round(acc_clm, 4)
    print(f"   ECE: {ece_clm:.4f} | ACC: {acc_clm:.4f}")

    # Task 2: Facts Generation
    print(f"\n📊 Task 2: Facts Generation (n={n_facts})...")
    ece_facts, acc_facts, n_facts_actual = evaluate_facts(
        model, tokenizer, trex_df, n_samples=n_facts
    )
    results['ece_facts'] = round(ece_facts, 4)
    results['acc_facts'] = round(acc_facts, 4)
    print(f"   ECE: {ece_facts:.4f} | ACC: {acc_facts:.4f}")

    # Task 3: MMLU
    print(f"\n📊 Task 3: MMLU (n={n_mmlu})...")
    ece_mmlu, acc_mmlu, n_mmlu_actual = evaluate_mmlu(
        model, tokenizer, mmlu_df, mmlu_val_df, n_samples=n_mmlu
    )
    results['ece_mmlu'] = round(ece_mmlu, 4)
    results['acc_mmlu'] = round(acc_mmlu, 4)
    print(f"   ECE: {ece_mmlu:.4f} | ACC: {acc_mmlu:.4f}")

    # Cleanup GPU memory
    print("\n🧹 Cleaning up GPU memory...")
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print(f"   GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    return results

print("\n✅ Everything ready!")
print(f"\n📋 Phase 1 models to evaluate:")
for m in PHASE1_MODELS:
    print(f"   → {m}")
print(f"\n📊 Evaluation samples:")
print(f"   CLM:   1000 samples")
print(f"   Facts: 1000 samples")
print(f"   MMLU:   500 samples")

📂 Loading datasets from Drive...
✅ PILE loaded:      50,000 samples
✅ T-REx loaded:     50,000 samples
✅ MMLU test:        14,042 samples
✅ MMLU validation:  1,531 samples

✅ Everything ready!

📋 Phase 1 models to evaluate:
   → EleutherAI/pythia-70m
   → EleutherAI/pythia-160m
   → EleutherAI/pythia-410m
   → EleutherAI/pythia-1b
   → EleutherAI/pythia-1.4b

📊 Evaluation samples:
   CLM:   1000 samples
   Facts: 1000 samples
   MMLU:   500 samples


In [34]:
import torch
import numpy as np
import gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

base_path = '/content/drive/MyDrive/llm_calibration'
results_path = f'{base_path}/results/phase1_results.csv'

# Models to run
models_to_run = [
    ("EleutherAI/pythia-70m",  70.4),
    ("EleutherAI/pythia-160m", 160),
    ("EleutherAI/pythia-410m", 410),
]

all_results = []

for model_name, param_size in models_to_run:
    print(f"\n{'='*60}")
    print(f"🤖 Model: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map='auto'
    )
    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"✅ Loaded! Params: {n_params:.0f}M | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Task 1: CLM
    print(f"\n📊 Task 1: CLM (n=30000)...")
    ece_clm, acc_clm, _ = evaluate_clm_batched(
        model, tokenizer, pile_df, n_samples=30000, batch_size=16
    )
    print(f"   ECE: {ece_clm:.4f} | ACC: {acc_clm:.4f}")

    # Task 2: Facts
    print(f"\n📊 Task 2: Facts (n=30000)...")
    ece_facts, acc_facts, _ = evaluate_facts_batched(
        model, tokenizer, trex_df, n_samples=30000, batch_size=32
    )
    print(f"   ECE: {ece_facts:.4f} | ACC: {acc_facts:.4f}")

    # Task 3: MMLU
    print(f"\n📊 Task 3: MMLU (n=500)...")
    ece_mmlu, acc_mmlu, _ = evaluate_mmlu_batched(
        model, tokenizer, mmlu_df, mmlu_val_df, n_samples=500
    )
    print(f"   ECE: {ece_mmlu:.4f} | ACC: {acc_mmlu:.4f}")

    # Store
    result = {
        'model': model_name,
        'revision': 'final',
        'n_params_M': round(n_params, 1),
        'ece_clm':   round(ece_clm, 4),
        'acc_clm':   round(acc_clm, 4),
        'ece_facts': round(ece_facts, 4),
        'acc_facts': round(acc_facts, 4),
        'ece_mmlu':  round(ece_mmlu, 4),
        'acc_mmlu':  round(acc_mmlu, 4)
    }
    all_results.append(result)

    # Save after each model
    pd.DataFrame(all_results).to_csv(results_path, index=False)
    print(f"💾 Saved to Drive!")

    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🧹 GPU cleared: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Final table
print(f"\n{'='*60}")
print(f"📋 RESULTS SO FAR:")
print(f"{'='*60}")
print(pd.DataFrame(all_results)[['model','n_params_M',
                                  'ece_clm','acc_clm',
                                  'ece_facts','acc_facts',
                                  'ece_mmlu','acc_mmlu']].to_string(index=False))


🤖 Model: EleutherAI/pythia-70m


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Loaded! Params: 70M | GPU: 0.59 GB

📊 Task 1: CLM (n=30000)...


CLM: 100%|██████████| 1875/1875 [07:01<00:00,  4.45it/s]


   ECE: 0.0070 | ACC: 0.4228

📊 Task 2: Facts (n=30000)...


...

   ECE: 0.0037 | ACC: 0.5209

📊 Task 2: Facts (n=30000)...


Facts: 100%|██████████| 938/938 [11:40<00:00,  1.34it/s]


   ECE: 0.0828 | ACC: 0.0119

📊 Task 3: MMLU (n=500)...


MMLU: 100%|██████████| 63/63 [01:03<00:00,  1.00s/it]


   ECE: 0.2289 | ACC: 0.2740
💾 Saved to Drive!
🧹 GPU cleared: 0.31 GB

📋 RESULTS SO FAR:
                 model  n_params_M  ece_clm  acc_clm  ece_facts  acc_facts  ece_mmlu  acc_mmlu
 EleutherAI/pythia-70m        70.4   0.0070   0.4228     0.0953     0.0000    0.1135     0.274
EleutherAI/pythia-160m       162.3   0.0053   0.4727     0.0923     0.0001    0.2254     0.274
EleutherAI/pythia-410m       405.3   0.0037   0.5209     0.0828     0.0119    0.2289     0.274


In [41]:
import torch
import numpy as np
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ============================================================
# CORRECTED MMLU FUNCTION — FULL SEQUENCE SCORING
# ============================================================

def evaluate_mmlu_corrected(model, tokenizer, mmlu_df, mmlu_val_df, n_samples=None, n_shot=5, max_length=1024, batch_size=8):
    """
    MMLU with 5-shot examples and PROPER full-sequence scoring.
    Computes log-probability of the full "Answer: X" sequence.
    """
    model.eval()
    all_confidences = []
    all_accuracies = []

    # Prepare few-shot examples
    few_shot_examples = mmlu_val_df.sample(n=n_shot, random_state=SEED)
    few_shot_text = ""
    for _, ex in few_shot_examples.iterrows():
        choices = ex['choices']
        correct_idx = ex['answer']

        few_shot_text += f"Question: {ex['question']}\n"
        few_shot_text += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
        few_shot_text += f"Answer: {chr(65 + correct_idx)}\n\n"

    # Use full dataset or sample
    if n_samples is not None:
        samples = mmlu_df.sample(n=n_samples, random_state=SEED)
    else:
        samples = mmlu_df

    total = len(samples)

    with torch.no_grad():
        for i in tqdm(range(0, total, batch_size), desc="MMLU Corrected"):
            batch_samples = samples.iloc[i:i+batch_size]

            for idx, row in batch_samples.iterrows():
                choices = row['choices']
                correct_idx = row['answer']  # 0-3 index

                # Build prompt WITHOUT the answer
                prompt = few_shot_text
                prompt += f"Question: {row['question']}\n"
                prompt += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
                prompt += "Answer:"

                # Tokenize the prompt once
                prompt_ids = tokenizer.encode(
                    prompt, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(device)

                # Score each option by computing NLL of " X" appended to prompt
                option_tokens = [' A', ' B', ' C', ' D']
                option_scores = []

                for opt in option_tokens:
                    # Tokenize the option
                    opt_ids = tokenizer.encode(opt, add_special_tokens=False)
                    opt_ids = torch.tensor(opt_ids, dtype=torch.long).unsqueeze(0).to(device)

                    # Concatenate prompt + option
                    full_ids = torch.cat([prompt_ids, opt_ids], dim=1)

                    # Create labels: -100 for prompt (ignore), keep for option
                    labels = torch.full_like(full_ids, -100)
                    labels[0, -len(opt_ids[0]):] = opt_ids[0]

                    # Forward pass with labels to get NLL
                    outputs = model(full_ids, labels=labels)
                    nll = outputs.loss.item()  # average NLL over option tokens

                    # Convert to log-probability (higher = better)
                    # Multiply by number of option tokens to get total log-prob
                    log_prob = -nll * len(opt_ids[0])
                    option_scores.append(log_prob)

                # Softmax over the 4 scores
                scores_tensor = torch.tensor(option_scores)
                probs = torch.softmax(scores_tensor, dim=-1)

                pred_idx = torch.argmax(probs).item()
                confidence = probs[pred_idx].item()

                is_correct = int(pred_idx == correct_idx)

                all_confidences.append(confidence)
                all_accuracies.append(is_correct)

    conf = np.array(all_confidences)
    acc_arr = np.array(all_accuracies)

    ece = compute_ece(conf, acc_arr)
    mean_acc = np.mean(acc_arr)

    return ece, mean_acc, len(conf)


# ============================================================
# RUN CORRECTED MMLU ON ALL 3 MODELS
# ============================================================

models_to_run = [
    ("EleutherAI/pythia-70m",  70.4),
    ("EleutherAI/pythia-160m", 162.3),
    ("EleutherAI/pythia-410m", 405.3),
]

# Old results for comparison
old_results = {
    "EleutherAI/pythia-70m":  {"ece": 0.1324, "acc": 0.2551},
    "EleutherAI/pythia-160m": {"ece": 0.2443, "acc": 0.2551},
    "EleutherAI/pythia-410m": {"ece": 0.2478, "acc": 0.2551},
}

print("="*60)
print("🔄 RE-RUNNING MMLU — CORRECTED SEQUENCE SCORING")
print("="*60)

for model_name, param_size in models_to_run:
    print(f"\n{'='*60}")
    print(f"🤖 Model: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map='auto'
    )
    model.eval()
    print(f"✅ Loaded! GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Run corrected MMLU
    ece_mmlu, acc_mmlu, n = evaluate_mmlu_corrected(
        model, tokenizer, mmlu_df, mmlu_val_df, n_samples=None
    )

    old = old_results[model_name]

    print(f"\n📊 MMLU Results (Corrected - Full Dataset):")
    print(f"   Samples:  {n}")
    print(f"   ECE:      {ece_mmlu:.4f}  (was {old['ece']:.4f})")
    print(f"   ACC:      {acc_mmlu:.4f}  (was {old['acc']:.4f})")

    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🧹 GPU cleared: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"\n{'='*60}")
print("✅ Corrected MMLU evaluation complete!")
print(f"{'='*60}")

🔄 RE-RUNNING MMLU — CORRECTED SEQUENCE SCORING

🤖 Model: EleutherAI/pythia-70m


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Loaded! GPU: 1.09 GB


MMLU Corrected:   0%|          | 0/1756 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import torch
import numpy as np
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ============================================================
# CORRECTED MMLU FUNCTION — FULL SEQUENCE SCORING
# ============================================================

def evaluate_mmlu_corrected(model, tokenizer, mmlu_df, mmlu_val_df, n_samples=None, n_shot=5, max_length=1024, batch_size=8):
    """
    MMLU with 5-shot examples and PROPER full-sequence scoring.
    Computes log-probability of the full "Answer: X" sequence.
    """
    model.eval()
    all_confidences = []
    all_accuracies = []

    # Prepare few-shot examples
    few_shot_examples = mmlu_val_df.sample(n=n_shot, random_state=SEED)
    few_shot_text = ""
    for _, ex in few_shot_examples.iterrows():
        choices = ex['choices']
        correct_idx = ex['answer']

        few_shot_text += f"Question: {ex['question']}\n"
        few_shot_text += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
        few_shot_text += f"Answer: {chr(65 + correct_idx)}\n\n"

    # Use full dataset or sample
    if n_samples is not None:
        samples = mmlu_df.sample(n=n_samples, random_state=SEED)
    else:
        samples = mmlu_df

    total = len(samples)

    with torch.no_grad():
        for i in tqdm(range(0, total, batch_size), desc="MMLU Corrected"):
            batch_samples = samples.iloc[i:i+batch_size]

            for idx, row in batch_samples.iterrows():
                choices = row['choices']
                correct_idx = row['answer']  # 0-3 index

                # Build prompt WITHOUT the answer
                prompt = few_shot_text
                prompt += f"Question: {row['question']}\n"
                prompt += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
                prompt += "Answer:"

                # Tokenize the prompt once
                prompt_ids = tokenizer.encode(
                    prompt, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(device)

                # Score each option by computing NLL of " X" appended to prompt
                option_tokens = [' A', ' B', ' C', ' D']
                option_scores = []

                for opt in option_tokens:
                    # Tokenize the option
                    opt_ids = tokenizer.encode(opt, add_special_tokens=False)
                    opt_ids = torch.tensor(opt_ids, dtype=torch.long).unsqueeze(0).to(device)

                    # Concatenate prompt + option
                    full_ids = torch.cat([prompt_ids, opt_ids], dim=1)

                    # Create labels: -100 for prompt (ignore), keep for option
                    labels = torch.full_like(full_ids, -100)
                    labels[0, -len(opt_ids[0]):] = opt_ids[0]

                    # Forward pass with labels to get NLL
                    outputs = model(full_ids, labels=labels)
                    nll = outputs.loss.item()  # average NLL over option tokens

                    # Convert to log-probability (higher = better)
                    # Multiply by number of option tokens to get total log-prob
                    log_prob = -nll * len(opt_ids[0])
                    option_scores.append(log_prob)

                # Softmax over the 4 scores
                scores_tensor = torch.tensor(option_scores)
                probs = torch.softmax(scores_tensor, dim=-1)

                pred_idx = torch.argmax(probs).item()
                confidence = probs[pred_idx].item()

                is_correct = int(pred_idx == correct_idx)

                all_confidences.append(confidence)
                all_accuracies.append(is_correct)

    conf = np.array(all_confidences)
    acc_arr = np.array(all_accuracies)

    ece = compute_ece(conf, acc_arr)
    mean_acc = np.mean(acc_arr)

    return ece, mean_acc, len(conf)


# ============================================================
# QUICK TEST ON ALL 3 MODELS WITH 3000 SAMPLES
# ============================================================

models_to_run = [
    ("EleutherAI/pythia-70m",  70.4),
    ("EleutherAI/pythia-160m", 162.3),
    ("EleutherAI/pythia-410m", 405.3),
]

print("="*60)
print("🧪 QUICK TEST — CORRECTED MMLU (n=3000)")
print("="*60)

for model_name, param_size in models_to_run:
    print(f"\n{'='*60}")
    print(f"🤖 Model: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map='auto'
    )
    model.eval()
    print(f"✅ Loaded! GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # Run corrected MMLU with 3000 samples
    ece_mmlu, acc_mmlu, n = evaluate_mmlu_corrected(
        model, tokenizer, mmlu_df, mmlu_val_df, n_samples=3000
    )

    print(f"\n📊 MMLU Results (Corrected - n=3000):")
    print(f"   ECE: {ece_mmlu:.4f}")
    print(f"   ACC: {acc_mmlu:.4f}")
    print(f"   Samples: {n}")

    # Check if accuracy > 25% (good sign)
    if acc_mmlu > 0.26:
        print(f"   ✅ Accuracy above random! Method likely working.")
    else:
        print(f"   ⚠️ Still near random. Method may still have issues.")

    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🧹 GPU cleared: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"\n{'='*60}")
print("✅ Quick test complete!")
print(f"{'='*60}")

🧪 QUICK TEST — CORRECTED MMLU (n=3000)

🤖 Model: EleutherAI/pythia-70m


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Loaded! GPU: 1.59 GB


MMLU Corrected: 100%|██████████| 375/375 [06:43<00:00,  1.08s/it]



📊 MMLU Results (Corrected - n=3000):
   ECE: 0.3274
   ACC: 0.2660
   Samples: 3000
   ✅ Accuracy above random! Method likely working.
🧹 GPU cleared: 0.52 GB

🤖 Model: EleutherAI/pythia-160m


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Loaded! GPU: 1.17 GB


MMLU Corrected:  95%|█████████▍| 355/375 [17:09<00:57,  2.89s/it]